# Five numbers

The run from **`train_five_params.py`**. The entire model is five scalars -- everything else is
frozen at zero, or is the identity $W_{ov}$.

| symbol | what it does | where it lives |
|---|---|---|
| $p$ | position ramp: $\mathrm{score}(i,j) = -p\,(i-j)$ | `p_ramp`, a scalar parameter |
| $a_1$ | layer 1 noise veto | `W_qk1[0:8, 4:8]`, one shared value |
| $a_2$ | layer 2 noise veto | `W_qk2[0:8, 4:8]`, one shared value |
| $b_2$ | induction | `W_qk2[0:4, 108:112]` diagonal |
| $c_2$ | state | `W_qk2[108:112, 108:112]` diagonal |

Verified on a trained checkpoint: writing those five numbers down and rebuilding all four
matrices from nothing else reproduces the checkpoint with `max|diff| = 0.000e+00`, including
all 4,950 position cells.

**The position block is not stored.** It is not a parameter any more, so `layer1.W_qk` on disk
holds only the 32 cells of $a_1$. `full_qk()` below rebuilds the effective matrix:

```
W1 = zeros(shape);  W1[mask] = row[dofidx];  W1[8:108, 8:108] += p * RAMP
```

For the **gradient** row the same function is fed $dL/dp$ instead of $p$, so those panels show
$g_p \times \mathrm{RAMP}$ -- the gradient's component along the ramp. The individual position
cells have no gradient of their own; there is only one number there.

**What to watch.** Every reduction before this one was gauge-fixing: the same model written
with fewer symbols. This one is not. The construction's ramp is piecewise linear with a **cap**
($+50\ +30\ +10\ -10\ -30$, then flat), and a pure line cannot bend. At $p \approx 10$ the score
at $d = 99$ is around $-1000$ logits, so anything past roughly $d = 10$ is annihilated. Whether
that cap was load-bearing is what this run measures -- and the earlier `tied4_` result, where GD
put $d = 2$ and $d = 3$ *above* $d = 1$, suggests the true shape is not monotone.

In [ ]:
import os, json, glob, re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.colors import SymLogNorm
from IPython.display import HTML, display
plt.rcParams['animation.embed_limit'] = 300

V, Vs, T = 8, 4, 100
D1, D2 = V + T, 2 * (V + T)          # 108, 216

RUN_PREFIX = 'tied5_'          # train_three_params.py tags its runs this way;
                               # train.py uses 'tied_', train_l1_two_parts.py 'tied2_'

def discover_run():
    '''the RUN_PREFIX run with the most epochs that has snapshots -- falls back to the newest'''
    cands = [d for d in glob.glob(f'runs/{RUN_PREFIX}*/')
             if glob.glob(os.path.join(d, 'snapmeta_*.json'))]
    if not cands:
        raise FileNotFoundError(
            f'no runs/{RUN_PREFIX}* has snapshots yet -- start train_three_params.py first.\n'
            f'  (runs present: {sorted(os.path.basename(d.rstrip("/")) for d in glob.glob("runs/*/"))})')
    def eps(d):
        m = re.search(r'_ep-(\d+)', d)
        return int(m.group(1)) if m else -1
    return max(cands, key=lambda d: (eps(d), os.path.getmtime(d))).rstrip('/')

# Pin the run by name. Set RUN_NAME = None to fall back to "the RUN_PREFIX run with the most
# epochs" -- but naming it means the notebook cannot silently switch runs mid-analysis.
RUN_NAME = 'tied5_opt-adam_lr-1.0_init-zeros_wov-fixed_blocks-relevant_readout-bias_ep-200'

if RUN_NAME:
    RUN = os.path.join('runs', RUN_NAME)
    if not glob.glob(f'{RUN}/snapmeta_*.json'):
        raise FileNotFoundError(
            f'{RUN} has no snapshots.\n  runs present: '
            f'{sorted(os.path.basename(d.rstrip("/")) for d in glob.glob("runs/*/"))}')
else:
    RUN = discover_run()
print(RUN, '\n')

def thresholds_present():
    '''read them off the files -- a run may use any --thresholds, not just 0.6/0.8/1.0'''
    out = []
    for p in glob.glob(f'{RUN}/snapmeta_x_*.json'):
        m = re.search(r'snapmeta_x_([0-9.]+)\.json$', p)
        if m:
            out.append(float(m.group(1)))
    return sorted(out)

THRS = thresholds_present()
if not THRS:
    raise FileNotFoundError(f'{RUN} has no snapmeta_x_*.json')
for _t in THRS:
    _m = json.load(open(f'{RUN}/snapmeta_x_{_t}.json'))
    _done = _m['w_rows_written'] >= _m['rows_allocated']
    print(f"  alpha={_t}: w={_m['w_rows_written']}/{_m['rows_allocated']}  g={_m['g_rows_written']}  "
          f"{'complete' if _done else 'STILL TRAINING -- cells below use what exists'}")

def aligned_losses(thr):
    '''train and test loss indexed by the weights they describe.

    train.py already evaluates BOTH after the step, so no shift is needed here -- unlike
    folder 18, where train_loss was the pre-step value and lagged test by one. `train_loss_pre`
    is kept in the metrics for exactly that comparison.

    Empty arrays come back when a threshold has not finished: metrics_*.json is written when
    the threshold ENDS, while snapmeta / w / g appear from step 0. Callers check .size.
    '''
    p = f'{RUN}/metrics_seed0_x_{thr}.json'
    if not os.path.exists(p):
        e = np.array([])
        return e, e, e, float('nan'), 0
    h = json.load(open(p))['history']
    return (np.asarray(h['step']), np.asarray(h['train_loss']), np.asarray(h['test_loss']),
            h['test_loss'][-1], h['step'][-1])

def load_dof(param, thr, kind='w'):
    '''(mask, dofidx, rows) for the new per-dof storage.

    A stored row holds ONE VALUE PER DEGREE OF FREEDOM, not one per cell -- layer 2 is three
    numbers. Rebuild any step k with   W = zeros(shape); W[mask] = rows[k][dofidx].
    '''
    mask = np.load(f'{RUN}/mask_{param}_x_{thr}.npy')
    dofidx = np.load(f'{RUN}/dofidx_{param}_x_{thr}.npy')
    rows = np.load(f'{RUN}/{kind}_{param}_x_{thr}.npy', mmap_mode='r')
    return mask, dofidx, rows

# The position block is NOT stored: it is p * RAMP, rebuilt here. RAMP[i, j] = -(i - j) for
# i > j (row = query position, col = key position) and 0 above the causal diagonal, so
# p * RAMP is exactly the score -p (i - j) the model applies.
_i = np.arange(T).reshape(-1, 1)
_j = np.arange(T).reshape(1, -1)
RAMP = np.where(_i - _j > 0, -(_i - _j), 0).astype(float)

def full_qk(param, mt, mask, dofidx, row, p_scalar):
    '''the effective matrix at one step: stored dofs, plus the ramp for layer 1.

    `p_scalar` is p for a weight row and dL/dp for a gradient row -- the gradient panels
    therefore show g_p * RAMP, the gradient's component along the ramp, since the individual
    position cells are not parameters and carry no gradient of their own.
    '''
    A = np.zeros(mt['shapes'][param])
    A[mask] = row[dofidx]
    if param == 'layer1.W_qk':
        A[V:D1, V:D1] += float(p_scalar) * RAMP
    return A

def group_map(meta, param, mask, dofidx):
    '''full-shape int array: which dof each cell belongs to.

    -1 means "not worth outlining": a dof that owns a single cell (every cell of the position
    block). Only dofs shared by two or more cells get a black outline in the panels.
    '''
    gid = np.full(meta['shapes'][param], -1, dtype=int)
    gid[mask] = dofidx
    cnt = np.bincount(dofidx, minlength=int(dofidx.max()) + 1)
    solo = np.flatnonzero(cnt <= 1)
    if solo.size:
        gid[np.isin(gid, solo)] = -1
    return gid

def panel(ax, A, title, vmax, annot, gid=None):
    v = vmax if vmax > 0 else 1.0
    ax.imshow(A, cmap='RdBu_r', interpolation='nearest',
              norm=SymLogNorm(linthresh=v / 1e3, linscale=0.6, vmin=-v, vmax=v, base=10))
    if annot:
        lab = [['0' if z == 0 else (f'{z:.3g}' if abs(z) >= 0.01 else f'{z:.1e}') for z in row]
               for row in A]
        wid = max(len(x) for row in lab for x in row)
        fs = (12.5 if A.shape[0] <= Vs else 6.4) * min(1.0, 5.0 / wid)
        for i in range(A.shape[0]):
            for j in range(A.shape[1]):
                ax.text(j, i, lab[i][j], ha='center', va='center', fontsize=fs,
                        color='white' if abs(A[i, j]) > v / 3 else 'black')
        if A.shape[0] <= Vs:
            ax.set_xticks(range(A.shape[1])); ax.set_yticks(range(A.shape[0]))
            ax.set_xticklabels([f'k{q}' for q in range(A.shape[1])], fontsize=8)
            ax.set_yticklabels([f'q{q}' for q in range(A.shape[0])], fontsize=8)
            ax.tick_params(length=0)
        else:
            ax.set_xticks([]); ax.set_yticks([])
    else:
        ax.set_xticks([]); ax.set_yticks([])
    if gid is not None:                       # outline the tie groups: one parameter each
        for i in range(gid.shape[0]):
            for j in range(gid.shape[1]):
                if j + 1 < gid.shape[1] and gid[i, j] != gid[i, j + 1]:
                    ax.plot([j + .5, j + .5], [i - .5, i + .5], color='k', lw=2.2)
                if i + 1 < gid.shape[0] and gid[i, j] != gid[i + 1, j]:
                    ax.plot([j - .5, j + .5], [i + .5, i + .5], color='k', lw=2.2)
    ax.set_title(f'{title}\n$\\|\\cdot\\|_F$ {np.linalg.norm(A):.4g}', fontsize=9)

## The animation

`ANIM_FROM` / `ANIM_TO` are epochs, both inclusive. Colour scales are fixed across the frames
*shown*, so a late window is scaled to itself — which is what makes small late changes visible,
and also means two windows are not comparable by colour.

In [ ]:
ANIM_THR  = 0.6      # any alpha listed above
ANIM_FROM = 0        # first epoch  (inclusive)
ANIM_TO   = 50       # last  epoch  (inclusive)
#   51 frames of this 6-panel figure is ~17 MB inline. The cell prints the size; shorten the
#   range or lower ANIM_DPI if the notebook gets heavy.
LOSS_ROW_H = 1.24     # height of the loss row, in units of one matrix row (0.62 = old)
ANIM_DPI   = 88       # six panels wide now -- drop this if the notebook feels heavy

# A 10x10 window from the MIDDLE of layer 1's 100x100 position block, blown up to a panel
# of its own and placed third, next to the full block. The full 100x100 is 9,900 near-identical
# ramp rows in which nothing is legible; this window is where the ramp actually lives -- it
# straddles the causal diagonal, so the decay with distance is visible cell by cell.
ZOOM_AT, ZOOM_N = 45, 10                      # position block rows/cols [45:55]
ZOOM = slice(V + ZOOM_AT, V + ZOOM_AT + ZOOM_N)

# the six panels: (param, row slice, col slice, label, annotate?)
CUT = [('layer1.W_qk', slice(0, V),        slice(0, V),        'L1 tok x tok',     True),
       ('layer1.W_qk', slice(V, D1),       slice(V, D1),       'L1 pos x pos',     False),
       ('layer1.W_qk', ZOOM,               ZOOM,
        f'L1 pos x pos  [{ZOOM_AT}:{ZOOM_AT + ZOOM_N}]  zoom',                     True),
       ('layer2.W_qk', slice(0, V),        slice(0, V),        'L2 A  noise suppr.', True),
       ('layer2.W_qk', slice(0, Vs),       slice(D1, D1 + Vs), 'L2 B  induction',  True),
       ('layer2.W_qk', slice(D1, D1 + Vs), slice(D1, D1 + Vs), 'L2 C  state',      True)]

def animate(thr, first=0, last=50, interval=450):
    mp = f'{RUN}/snapmeta_x_{thr}.json'
    if not os.path.exists(mp):
        raise FileNotFoundError(f'no snapshots for alpha={thr} in {RUN}')
    mt = json.load(open(mp))
    kmax = min(mt['w_rows_written'] - 1, mt['g_rows_written'] - 1)   # step k needs w[k] AND g[k]
    a, b = max(0, int(first)), min(int(last), kmax)
    if b < a:
        raise ValueError(f'empty range [{first}, {last}] -> [{a}, {b}]; '
                         f'this run has steps 0..{kmax} for alpha={thr}')
    if b < int(last):
        print(f'alpha={thr}: only steps 0..{kmax} recorded so far -> showing [{a}, {b}]')
    n = b - a + 1
    print(f'alpha={thr}: epochs {a}..{b}   ({n} frames)')

    Mk, DOF, Wf, Gf = {}, {}, {}, {}
    for p in mt['params']:
        Mk[p], DOF[p], rw = load_dof(p, thr, 'w')
        Wf[p] = np.asarray(rw[a:b + 1])
        Gf[p] = np.asarray(load_dof(p, thr, 'g')[2][a:b + 1])
    GID = {p: group_map(mt, p, Mk[p], DOF[p]) for p in mt['params']}

    def blocks(src, p, rs, cs):
        '''rebuild the block from the stored dofs, adding p * RAMP for layer 1'''
        out = np.empty((n, rs.stop - rs.start, cs.stop - cs.start))
        for k in range(n):
            A = full_qk(p, mt, Mk[p], DOF[p], src[p][k], src['p_ramp'][k][0])
            out[k] = A[rs, cs]
        return out

    Ws = [blocks(Wf, p, rs, cs) for p, rs, cs, _, _ in CUT]
    Gs = [blocks(Gf, p, rs, cs) for p, rs, cs, _, _ in CUT]
    vw = [float(np.abs(x).max()) for x in Ws]
    vg = [float(np.abs(x).max()) for x in Gs]
    gids = [GID[p][rs, cs] if an else None for p, rs, cs, _, an in CUT]

    st, tr, te, fin, nlast = aligned_losses(thr)
    st, tr, te = st[a:b + 1], tr[a:b + 1], te[a:b + 1]

    nb_ = len(CUT)
    # Three rows. The loss curve is no longer a narrow column on the right: it gets its own
    # row underneath both matrix rows, stretched across the full width, with the same panel
    # height it had before (LOSS_ROW_H is in units of one matrix row).
    f = plt.figure(figsize=(4.6 * nb_, 9.2 + 4.6 * LOSS_ROW_H + 1.55), dpi=ANIM_DPI)
    g = f.add_gridspec(3, nb_, height_ratios=[1, 1, LOSS_ROW_H], wspace=0.10, hspace=0.16)
    ax = np.array([[f.add_subplot(g[r, c_]) for c_ in range(nb_)] for r in range(2)])
    al = f.add_subplot(g[2, :])
    f.subplots_adjust(top=0.92, bottom=0.06, left=0.025, right=0.99)

    def dr(i_):
        ep = a + i_
        for a_ in ax.ravel():
            a_.clear()
        for j_, (_, _, _, ttl, an) in enumerate(CUT):
            # the position cells are not parameters, so their "gradient" is g_p * RAMP
            gt = ttl + ('  ($g_p\\times$RAMP)' if 'pos x pos' in ttl else '')
            panel(ax[0][j_], Ws[j_][i_], f'W  {ttl}', vw[j_], an, gids[j_])
            panel(ax[1][j_], Gs[j_][i_], f'grad  {gt}', vg[j_], an, gids[j_])
        al.clear()
        if not st.size:                      # threshold still running: no metrics file yet
            al.text(0.5, 0.5, f'metrics_seed0_x_{thr}.json not written yet\n'
                              f'(it appears when this threshold finishes)',
                    transform=al.transAxes, ha='center', va='center',
                    fontsize=12, color='0.45')
            al.set_xticks([]); al.set_yticks([])
            f.suptitle(f'$\\alpha$={thr}   epoch {ep}   (range {a}..{b})   '
                       f'top: $W$;  middle: $g[{ep}]$   '
                       f'[black outlines = one tied parameter]', fontsize=13)
            return []
        al.plot(st, tr, color='#1f77b4', lw=1.8, label='train')
        al.plot(st, te, color='#ff7f0e', lw=1.4, ls='--', label='test')
        al.axvline(ep, color='k', lw=1.1, alpha=0.65)
        al.plot([ep], [tr[i_]], 'o', color='#1f77b4', ms=8, zorder=5)
        al.plot([ep], [te[i_]], 'o', color='#ff7f0e', ms=8, zorder=5)
        al.annotate(f'epoch {ep}\ntrain {tr[i_]:.5f}\ntest  {te[i_]:.5f}',
                    xy=(0.96, 0.95), xycoords='axes fraction', ha='right', va='top',
                    fontsize=8.5, family='monospace',
                    bbox=dict(boxstyle='round,pad=0.4', fc='white', ec='0.7', alpha=0.9))
        al.set_xlim(a - 0.5, b + 0.5)
        al.set_xlabel('optimizer step (= epoch)', fontsize=9)
        al.set_ylabel('cross entropy', fontsize=9); al.tick_params(labelsize=8)
        al.set_title(f'loss over epochs {a}..{b}\n(run ends at {fin:.5f} after {nlast})',
                     fontsize=9)
        al.grid(alpha=0.3); al.legend(fontsize=8, loc='lower left')
        note = '   --  zero init: W all zero, and layer 1 has NO gradient yet' if ep == 0 else ''
        f.suptitle(f'$\\alpha$={thr}   epoch {ep}   (range {a}..{b})   '
                   f'top: $W$ before update {ep+1};  middle: $g[{ep}]$ at those weights;  '
                   f'bottom: the loss'
                   f'   [black outlines = one tied parameter]{note}', fontsize=13)
        return []

    an_ = animation.FuncAnimation(f, dr, frames=n, interval=interval, blit=False)
    plt.close(f)
    html = an_.to_jshtml()
    print(f'  animation {len(html) / 1e6:.1f} MB   ({nb_} panels x 2 rows + loss row)')
    return HTML(html)

display(animate(ANIM_THR, ANIM_FROM, ANIM_TO))

## The five numbers, on their own

Same data as the panels above, read as scalars.

In [ ]:
# ------------------------------------------------------------------ the five numbers
# Read in LOGIT units (stored entry / sqrt(108)) -- what the softmax actually sees.
SDq = D1 ** 0.5
SPEC = [('p_ramp',      'p_ramp',              '$p$   position slope', '#8a4fbf'),
        ('layer1.W_qk', 'L1.A.knoi',           '$a_1$  L1 noise veto', '#2f6fc2'),
        ('layer2.W_qk', 'L2.A.knoi',           '$a_2$  L2 noise veto', '#66aadd'),
        ('layer2.W_qk', 'L2.B.induction.diag', '$b_2$  induction',     '#c2492f'),
        ('layer2.W_qk', 'L2.C.state.diag',     '$c_2$  state',         '#1a8f4a')]

def five(thr):
    """(steps, {label: values in logits}) for one threshold, truncated to what is written"""
    mt = json.load(open(f'{RUN}/snapmeta_x_{thr}.json'))
    k = mt['w_rows_written']
    steps = np.asarray(mt['w_steps'][:k])
    out = {}
    for param, gname, lbl, _ in SPEC:
        gn = mt['tie_group_names'][param]
        col = gn.index(gname) if gname in gn else 0     # p_ramp has no tie group: dof 0
        out[lbl] = np.asarray(load_dof(param, thr, 'w')[2][:k, col]) / SDq
    return steps, out

fig, axes = plt.subplots(1, len(THRS), figsize=(6.2 * len(THRS), 4.6),
                         sharey=True, squeeze=False)
print(f'{"alpha":>6} ' + ''.join(f'{l.split("  ")[-1][:13]:>15}' for _, _, l, _ in SPEC)
      + f'{"b2 - c2":>11}{"test CE":>14}')
for ax, thr in zip(axes[0], THRS):
    steps, vals = five(thr)
    for param, gname, lbl, col in SPEC:
        ax.plot(steps, vals[lbl], color=col, lw=2.1, label=lbl)
    ax.axhline(0, color='0.7', lw=0.8)
    ax.set_title(f'$\\alpha$ = {thr}   ({len(steps)} epochs recorded)', fontsize=11)
    ax.set_xlabel('epoch'); ax.grid(alpha=0.25)
    *_, fin, _ = aligned_losses(thr)
    ce = 'still running' if np.isnan(fin) else f'{fin:.5f}'
    b2c2 = vals['$b_2$  induction'][-1] - vals['$c_2$  state'][-1]
    print(f'{thr:>6} ' + ''.join(f'{vals[l][-1]:>15.4f}' for _, _, l, _ in SPEC)
          + f'{b2c2:>11.4f}{ce:>14}')
axes[0][0].set_ylabel('logits   (entry / $\\sqrt{108}$)')
axes[0][0].legend(fontsize=10, loc='lower left', framealpha=0.95)
fig.suptitle('The whole model, over training', fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()

print('\nb2 - c2 is the SIGNAL-QUERY MARGIN: the logit gap between the key whose state matches')
print('the current token and the rival key whose state matches the PREVIOUS state.')
print('Careful with conventions. Here the off-diagonals are ZERO, so b2 and c2 ARE the block')
print('contrasts (diag - off) and the margin is b2 - c2. In the +b/-b writing used earlier the')
print('same margin is 2(b - c). Either way it is the CONTRAST difference, and it saturates')
print('around 4 logits: sigmoid(4) = 0.982 of the attention on the right keys.')